# Encoder Transfer OPT

In [ ]:
import os, random, numpy as np, pandas as pd, torch
from typing import List, Tuple, Dict, DefaultDict
from collections import defaultdict
from sklearn.metrics import f1_score, accuracy_score
from transformers import AutoTokenizer, AutoModel

DATASETS = ["hatexplain", "olid", "sbic", "ihc"]
SEEDS    = list(range(10))  
MODEL_PATTERN     = "iproskurina/opt-125m-{ds}-s{seed}"
CSV_PATTERN_TRAIN = "{ds}_train.csv"
CSV_PATTERN_TEST  = "{ds}_test.csv"

TEXT_COL  = "sentence"
LABEL_COL = "label"

BATCH_SIZE  = 8
MAX_LENGTH  = 500
USE_FP16    = False
MAX_PROTOS_PER_CLASS = 500

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def normalize_labels(series: pd.Series) -> pd.Series:
    def _to_int(x):
        if isinstance(x, str):
            xl = x.strip().lower()
            try: return int(x)
            except: raise ValueError(f"Unrecognized label: {x}")
        if isinstance(x, (int, np.integer)) and x in (0,1): return int(x)
        raise ValueError(f"Unsupported label value: {x}")
    return series.apply(_to_int)

class TextDS(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tok, max_len):
        self.texts, self.labels, self.tok, self.max_len = texts, labels, tok, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(str(self.texts[i]),
                       truncation=True, padding="max_length",
                       max_length=self.max_len, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(int(self.labels[i])).long()
        return item

def make_loader(texts, labels, tok, max_len, bs, shuffle=False):
    return torch.utils.data.DataLoader(
        TextDS(texts, labels, tok, max_len),
        batch_size=bs, shuffle=shuffle, pin_memory=torch.cuda.is_available()
    )

@torch.no_grad()
def collect_last_token(model, loader, device) -> Tuple[np.ndarray, List[int]]:
    """
    Decoder-only (OPT): last hidden layer @ last non-padding token.
    Padding-side agnostic.
    """
    model.eval()
    feats, ys = [], []
    for batch in loader:
        ids = batch["input_ids"].to(device)
        att = batch["attention_mask"].to(device) 
        ys.extend(batch["labels"].tolist())

        out = model(input_ids=ids, attention_mask=att, output_hidden_states=True, return_dict=True)
        h_last = out.hidden_states[-1]  # (B, T, D)
        last_idx = att.size(1) - 1 - torch.argmax(att.flip(1), dim=1)  # (B,)
        reps = h_last[torch.arange(h_last.size(0), device=h_last.device), last_idx, :]  # (B, D)
        feats.append(reps.detach().float().cpu().numpy())

    feats = np.concatenate(feats, axis=0) if feats else np.zeros((0, model.config.hidden_size))
    return feats, ys

def l2_normalize(x: np.ndarray, axis: int = -1, eps: float = 1e-8) -> np.ndarray:
    n = np.linalg.norm(x, axis=axis, keepdims=True)
    return x / (n + eps)

def build_class_means(feats: np.ndarray, labels: List[int]) -> Dict[int, np.ndarray]:
    y = np.array(labels)
    class_means = {}
    D = feats.shape[1] if feats.ndim == 2 else 0
    for c in (0, 1):
        fc = feats[y == c]
        if fc.size == 0:
            class_means[c] = np.zeros((D,), dtype=np.float32)
        else:
            fc = l2_normalize(fc, axis=1)
            mu = fc.mean(axis=0)
            mu = l2_normalize(mu[None, :], axis=1)[0]
            class_means[c] = mu
    return class_means

def cosine_classify(x: np.ndarray, p0: np.ndarray, p1: np.ndarray) -> np.ndarray:
    x = l2_normalize(x, axis=1)
    p0 = p0 / (np.linalg.norm(p0) + 1e-8)
    p1 = p1 / (np.linalg.norm(p1) + 1e-8)
    s0 = (x @ p0); s1 = (x @ p1)
    return np.stack([s0, s1], axis=1).argmax(axis=1)

def load_csv(ds: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    tr = pd.read_csv(CSV_PATTERN_TRAIN.format(ds=ds))
    te = pd.read_csv(CSV_PATTERN_TEST.format(ds=ds))
    for df in (tr, te):
        df.dropna(subset=[TEXT_COL, LABEL_COL], inplace=True)
        df["label"] = normalize_labels(df[LABEL_COL])
        df["text"]  = df[TEXT_COL].astype(str)
    return tr, te

def fmt_mean_std(vals: List[float]) -> str:
    if not vals:
        return "n/a"
    m, s = np.mean(vals), np.std(vals)
    return f"{m*100:.2f}±{s*100:.2f}"


DATA = {ds: load_csv(ds) for ds in DATASETS} 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype  = torch.float16 if (USE_FP16 and torch.cuda.is_available()) else torch.float32

ResultsF1: DefaultDict[str, DefaultDict[str, DefaultDict[str, List[float]]]] = \
    defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
ResultsAcc: DefaultDict[str, DefaultDict[str, DefaultDict[str, List[float]]]] = \
    defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

for S in DATASETS:  # encoder family (fine-tuned on S)
    print(f"\n=== Encoder family: fine-tuned on {S.upper()} ===")
    for seed in SEEDS:
        set_seed(seed)
        model_name = MODEL_PATTERN.format(ds=S, seed=seed)
        try:
            tok = AutoTokenizer.from_pretrained(model_name, use_fast=False)
            if tok.pad_token_id is None and tok.eos_token_id is not None:
                tok.pad_token = tok.eos_token
            model = AutoModel.from_pretrained(model_name, output_hidden_states=True)
        except Exception as e:
            print(f"[WARN] Could not load {model_name}: {e}")
            continue

        if USE_FP16 and device.type == "cuda":
            model = model.half()
        model.to(device)
        model.eval()

        proto_means: Dict[str, Dict[int, np.ndarray]] = {}
        for P in DATASETS:
            train_P, _ = DATA[P]
            tr0 = train_P[train_P["label"] == 0]
            tr1 = train_P[train_P["label"] == 1]
            p0  = tr0.head(min(MAX_PROTOS_PER_CLASS, len(tr0))) if len(tr0) else tr0
            p1  = tr1.head(min(MAX_PROTOS_PER_CLASS, len(tr1))) if len(tr1) else tr1
            protos_df = pd.concat([p0, p1], ignore_index=True)

            if len(protos_df) == 0:
                
                D = model.config.hidden_size
                proto_means[P] = {0: np.zeros((D,), np.float32), 1: np.zeros((D,), np.float32)}
                continue

            proto_loader = make_loader(protos_df["text"].tolist(), protos_df["label"].tolist(),
                                       tok, MAX_LENGTH, BATCH_SIZE, shuffle=False)
            proto_feats, proto_labels = collect_last_token(model, proto_loader, device)
            proto_means[P] = build_class_means(proto_feats, proto_labels)
        test_cache: Dict[str, Tuple[np.ndarray, List[int]]] = {}
        for T in DATASETS:
            _, test_T = DATA[T]
            test_loader = make_loader(test_T["text"].tolist(), test_T["label"].tolist(),
                                      tok, MAX_LENGTH, BATCH_SIZE, shuffle=False)
            test_feats, test_labels = collect_last_token(model, test_loader, device)
            test_cache[T] = (test_feats, test_labels)

        for P in DATASETS:
            p0 = proto_means[P][0]; p1 = proto_means[P][1]
            for T in DATASETS:
                feats_T, labels_T = test_cache[T]
                if len(feats_T) == 0:
                    continue
                preds = cosine_classify(feats_T, p0, p1)
                acc = float(accuracy_score(labels_T, preds))
                f1m = float(f1_score(labels_T, preds, average="macro"))
                ResultsAcc[S][P][T].append(acc)
                ResultsF1[S][P][T].append(f1m)
                os.makedirs("predictions-opt-full-protos", exist_ok=True)
                out_df = pd.DataFrame({
                    "pred": preds,
                    "true": labels_T,
                })
                out_path = f"predictions-opt-full-protos/preds_{S}_s{seed}_proto{P}_to_{T}.csv.gz"
                out_df.to_csv(out_path, index=False, compression="gzip")

        del model
        torch.cuda.empty_cache()

    print("\nMacro-F1 (mean±std %, rows = prototype from P, cols = evaluated on T)")
    print("P→T\t" + "\t".join(DATASETS))
    for P in DATASETS:
        row = [P]
        for T in DATASETS:
            row.append(fmt_mean_std(ResultsF1[S][P][T]))
        print("\t".join(row))

    print("\nAccuracy (mean±std %, rows = prototype from P, cols = evaluated on T)")
    print("P→T\t" + "\t".join(DATASETS))
    for P in DATASETS:
        row = [P]
        for T in DATASETS:
            row.append(fmt_mean_std(ResultsAcc[S][P][T]))
        print("\t".join(row))

    print("\nLaTeX rows (F1):")
    for P in DATASETS:
        cells = [fmt_mean_std(ResultsF1[S][P][T]) for T in DATASETS]
        print(f"{S} (protos={P}) & " + " & ".join(cells) + r" \\")


# Encoder Transfer BERT

In [ ]:
MODEL_PATTERN     = "iproskurina/bert-base-cased-{ds}-s{seed}"
CSV_PATTERN_TRAIN = "{ds}_train.csv"
CSV_PATTERN_TEST  = "{ds}_test.csv"

TEXT_COL  = "sentence"
LABEL_COL = "label"
BATCH_SIZE  = 8
MAX_LENGTH  = 500
USE_FP16    = False
MAX_PROTOS_PER_CLASS = 500

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def normalize_labels(series: pd.Series) -> pd.Series:
    mapping = {"hate":1, "unsafe":1, "implicit":1, "implicit_hate":1, "implicit-hate":1,
               "non-hate":0, "nonhate":0, "non_hate":0, "neutral":0, "safe":0}
    def _to_int(x):
        if isinstance(x, str):
            xl = x.strip().lower()
            if xl in mapping: return mapping[xl]
            try: return int(x)
            except: raise ValueError(f"Unrecognized label: {x}")
        if isinstance(x, (int, np.integer)) and x in (0,1): return int(x)
        raise ValueError(f"Unsupported label value: {x}")
    return series.apply(_to_int)

class TextDS(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tok, max_len):
        self.texts, self.labels, self.tok, self.max_len = texts, labels, tok, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(
            str(self.texts[i]),
            truncation=True, padding="max_length",
            max_length=self.max_len, return_tensors="pt",
            add_special_tokens=True
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(int(self.labels[i])).long()
        return item

def make_loader(texts, labels, tok, max_len, bs, shuffle=False):
    return torch.utils.data.DataLoader(
        TextDS(texts, labels, tok, max_len),
        batch_size=bs, shuffle=shuffle, pin_memory=torch.cuda.is_available()
    )

@torch.no_grad()
def collect_last_cls(model, loader, device) -> Tuple[np.ndarray, List[int]]:
    """
    BERT-like encoders: last hidden layer @ [CLS] position (index 0).
    """
    model.eval()
    feats, ys = [], []
    for batch in loader:
        ids = batch["input_ids"].to(device)
        att = batch["attention_mask"].to(device)
        ys.extend(batch["labels"].tolist())

        out = model(input_ids=ids, attention_mask=att,
                    output_hidden_states=True, return_dict=True)
        h_last = out.hidden_states[-1]  # (B, T, D)
        cls_vecs = h_last[:, 0, :]      # (B, D)
        feats.append(cls_vecs.detach().float().cpu().numpy())

    feats = np.concatenate(feats, axis=0) if feats else np.zeros((0, model.config.hidden_size))
    return feats, ys

def l2_normalize(x: np.ndarray, axis: int = -1, eps: float = 1e-8) -> np.ndarray:
    n = np.linalg.norm(x, axis=axis, keepdims=True)
    return x / (n + eps)

def build_class_means(feats: np.ndarray, labels: List[int]) -> Dict[int, np.ndarray]:
    """
    Proper prototypes:
      1) L2-normalize each sample
      2) average within class
      3) L2-normalize the mean
    """
    y = np.array(labels)
    class_means = {}
    D = feats.shape[1] if feats.ndim == 2 else 0
    for c in (0, 1):
        fc = feats[y == c]
        if fc.size == 0:
            class_means[c] = np.zeros((D,), dtype=np.float32)
        else:
            fc = l2_normalize(fc, axis=1)
            mu = fc.mean(axis=0)
            mu = l2_normalize(mu[None, :], axis=1)[0]
            class_means[c] = mu
    return class_means

def cosine_classify(x: np.ndarray, p0: np.ndarray, p1: np.ndarray) -> np.ndarray:
    x = l2_normalize(x, axis=1)
    p0 = p0 / (np.linalg.norm(p0) + 1e-8)
    p1 = p1 / (np.linalg.norm(p1) + 1e-8)
    s0 = (x @ p0); s1 = (x @ p1)
    return np.stack([s0, s1], axis=1).argmax(axis=1)

def load_csv(ds: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    tr = pd.read_csv(CSV_PATTERN_TRAIN.format(ds=ds))
    te = pd.read_csv(CSV_PATTERN_TEST.format(ds=ds))
    for df in (tr, te):
        df.dropna(subset=[TEXT_COL, LABEL_COL], inplace=True)
        df["label"] = normalize_labels(df[LABEL_COL])
        df["text"]  = df[TEXT_COL].astype(str)
    return tr, te

def fmt_mean_std(vals: List[float]) -> str:
    if not vals:
        return "n/a"
    m, s = np.mean(vals), np.std(vals)
    return f"{m*100:.2f}±{s*100:.2f}"

DATA = {ds: load_csv(ds) for ds in DATASETS}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype  = torch.float16 if (USE_FP16 and torch.cuda.is_available()) else torch.float32

ResultsF1: DefaultDict[str, DefaultDict[str, DefaultDict[str, List[float]]]] = \
    defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
ResultsAcc: DefaultDict[str, DefaultDict[str, DefaultDict[str, List[float]]]] = \
    defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

for S in DATASETS:
    print(f"\n=== Encoder family: fine-tuned on {S.upper()} ===")
    for seed in SEEDS:
        set_seed(seed)
        model_name = MODEL_PATTERN.format(ds=S, seed=seed)
        try:
            tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)

            if tok.pad_token is None and tok.eos_token is not None:
                tok.pad_token = tok.eos_token
            model = AutoModel.from_pretrained(model_name, output_hidden_states=True)
        except Exception as e:
            print(f"[WARN] Could not load {model_name}: {e}")
            continue

        if USE_FP16 and device.type == "cuda":
            model = model.half()
        model.to(device)
        model.eval()

        proto_means: Dict[str, Dict[int, np.ndarray]] = {}
        for P in DATASETS:
            train_P, _ = DATA[P]
            tr0 = train_P[train_P["label"] == 0]
            tr1 = train_P[train_P["label"] == 1]
            p0  = tr0.head(MAX_PROTOS_PER_CLASS)
            p1  = tr1.head(MAX_PROTOS_PER_CLASS)
            protos_df = pd.concat([p0, p1], ignore_index=True)

            if len(protos_df) == 0:
                D = model.config.hidden_size
                proto_means[P] = {0: np.zeros((D,), np.float32), 1: np.zeros((D,), np.float32)}
                continue

            proto_loader = make_loader(
                protos_df["text"].tolist(), protos_df["label"].tolist(),
                tok, MAX_LENGTH, BATCH_SIZE, shuffle=False
            )
            proto_feats, proto_labels = collect_last_cls(model, proto_loader, device)
            proto_means[P] = build_class_means(proto_feats, proto_labels)

        test_cache: Dict[str, Tuple[np.ndarray, List[int]]] = {}
        for T in DATASETS:
            _, test_T = DATA[T]
            test_loader = make_loader(
                test_T["text"].tolist(), test_T["label"].tolist(),
                tok, MAX_LENGTH, BATCH_SIZE, shuffle=False
            )
            test_feats, test_labels = collect_last_cls(model, test_loader, device)
            test_cache[T] = (test_feats, test_labels)

        for P in DATASETS:
            p0 = proto_means[P][0]; p1 = proto_means[P][1]
            for T in DATASETS:
                feats_T, labels_T = test_cache[T]
                if len(feats_T) == 0:
                    continue
                preds = cosine_classify(feats_T, p0, p1)
                acc = float(accuracy_score(labels_T, preds))
                f1m = float(f1_score(labels_T, preds, average="macro"))
                ResultsAcc[S][P][T].append(acc)
                ResultsF1[S][P][T].append(f1m)
                os.makedirs("predictions-bert-full-protos", exist_ok=True)
                out_df = pd.DataFrame({
                    "pred": preds,
                    "true": labels_T,
                })
                out_path = f"predictions-bert-full-protos/preds_{S}_s{seed}_proto{P}_to_{T}.csv.gz"
                out_df.to_csv(out_path, index=False, compression="gzip")

        del model
        torch.cuda.empty_cache()

    print("\nMacro-F1 (mean±std %, rows = prototype from P, cols = evaluated on T)")
    print("P→T\t" + "\t".join(DATASETS))
    for P in DATASETS:
        row = [P]
        for T in DATASETS:
            row.append(fmt_mean_std(ResultsF1[S][P][T]))
        print("\t".join(row))

    print("\nAccuracy (mean±std %, rows = prototype from P, cols = evaluated on T)")
    print("P→T\t" + "\t".join(DATASETS))
    for P in DATASETS:
        row = [P]
        for T in DATASETS:
            row.append(fmt_mean_std(ResultsAcc[S][P][T]))
        print("\t".join(row))
    print("\nLaTeX rows (F1):")
    for P in DATASETS:
        cells = [fmt_mean_std(ResultsF1[S][P][T]) for T in DATASETS]
        print(f"{S} (protos={P}) & " + " & ".join(cells) + r" \\")
